### RaschPy model selection worked example

RaschPy provides three model-selection tools for choosing the simplest model structure that adequately fits the data:

1. **RSM vs PCM** (`model_selection()`, on either model): is a single shared threshold structure (RSM) adequate, or are independent per-item thresholds (PCM) required to explain the data?
2. **MFRM rater parameterisation** (`model_selection()`, on `MFRM`): of the eight rater representations (global, items, thresholds, bivector, matrix, centrality, pseudo_halo, bistretch), achieves the optimal balance between explanation and complexity?
3. **MFRM "mixed" model** (`per_rater_model_selection()`): rather than forcing every rater into the same parameterisation, assign each rater individually the model that achieves the optimal balance between explanation and complexity.

All three support likelihood-ratio, AIC, and BIC criteria. This notebook simulates data under a known structure in each case and checks whether these criteria correctly recover the generating structure(s).

Import `raschpy`:

In [ ]:
import raschpy as rp

#### RSM vs PCM

Simulate data generated under RSM and check whether `model_selection()` (available on either `RSM` or `PCM`) correctly prefers RSM:

In [ ]:
sim_rsm = rp.RSM_Sim(no_of_items=15, no_of_persons=1500, max_score=4, seed=1)
rsm = rp.RSM(sim_rsm)
rsm.calibrate()
rsm.model_selection(test='AIC')
rsm.model_comparison_rsm_pcm_aic_summary

Simulate data genuinely generated under PCM and check that `model_selection()` correctly prefers PCM:

In [ ]:
sim_pcm = rp.PCM_Sim(no_of_items=15, no_of_persons=2000, max_score_vector=[4] * 15, category_base=3, seed=1)
pcm = rp.PCM(sim_pcm)
pcm.calibrate()
pcm.model_selection(test='AIC')
pcm.model_comparison_rsm_pcm_aic_summary

`test='LR'` and `test='BIC'` are also available — results are stored as `model_comparison_rsm_pcm_{lr,aic,bic}_summary` respectively (plus `_preferred`, and for AIC with `aic_sig_test=True`, a relative-likelihood `_aic_p`).

#### MFRM rater-parameterisation model selection

Simulate data genuinely generated under the `items` rater parameterisation (a separate severity per rater x item combination), using `MFRM_Sim_Items`, and check that `model_selection()` (which by default calibrates all eight parameterisations and ranks them) correctly identifies it:

In [ ]:
sim_mfrm = rp.MFRM_Sim_Items(no_of_items=10, no_of_persons=500, no_of_facet_elements=8, max_score=4,
                             item_range=3, facet_range=2, seed=1)
mfrm = rp.MFRM(sim_mfrm)
mfrm.calibrate_global()   # any one calibration is enough to instantiate the model
mfrm.model_selection(test='AIC')
mfrm.model_comparison_mfrm_aic_summary.round(2)

In [ ]:
mfrm.model_comparison_mfrm_aic_preferred

By default, `model_selection()` calibrates and compares all eight rater parameterisations. Pass `models=` to restrict the comparison to a selected subset. For example, to test only whether `pseudo_halo`'s 2-parameter item-stretch restriction is needed over the full `items` parameterisation (the model this data was actually generated under, so `items` should win convincingly):

In [ ]:
mfrm.model_selection(test='LR', models=['items', 'pseudo_halo'])
mfrm.model_comparison_mfrm_lr_summary

The `models=` argument restricts the set of models under consideration to those passed as a list, so can be useful when you are interested in a specific subset of the model family, for example the 'stretch' models. Restricting to just the three stretch models also demonstrates `aic_sig_test`'s baseline logic when the standard beseline `global` is not in the set: the baseline becomes whichever model has the fewest free parameters instead -- here `centrality` and `pseudo_halo` are tied (both 2-parameter restrictions), so the tie is broken by lowest AIC:

In [ ]:
mfrm.model_selection(test='AIC', models=['centrality', 'pseudo_halo', 'bistretch'])
mfrm.model_comparison_mfrm_aic_summary

#### MFRM mixed rater model

Real rating designs rarely have every rater behaving with the same complexity; some might be simple, uniformly-shifted raters (`'global'`), while others show more complex behaviour that need more complex representations to capture. Rather than forcing one parameterisation on everyone, `per_rater_model_selection()` assigns each rater a representation individually. By default (`test='AIC'`) it derives all eight rater representations from a single matrix-model calibration and picks the minimum-AIC one per rater, guarded by a significance test against whichever active model has the fewest parameters (`aic_sig_test`) and an optional effect-size walk-back (`min_effect`); `test='LR'` instead uses a top-down testing ladder (matrix → bivector → items/thresholds → global) with an extra targeted stretch-model test at the end.

`per_rater_model_selection(anchors=[...])` additionally supports anchoring the matrix calibration to a chosen rater subset before testing — different anchor sets can result in different per-rater assignments, each of which is valid with respect to its own anchoring frame of reference, since the zero-sum constraint means a non-uniform rater can induce apparent non-uniformity in others. Anchored results are stored as `anchor_rater_models` / `anchor_facet_effects_mixed` / `anchor_per_rater_model_selection_table` / `anchor_per_rater_model_selection_counts`.

Simulate 8 raters: 4 behave as simple `'global'` raters with zero severity, and 4 show a genuine per-item "halo" pattern (systematically more lenient on some items, more severe on others) — i.e. genuinely `'items'`-parameterised behaviour. Anchor to the four raters with zero severity.

In [ ]:
max_score = 4
no_of_items = 8

flat = {f'Item_{i + 1}': [0] * max_score for i in range(no_of_items)}
halo = {f'Item_{i + 1}': [(((no_of_items - 1) / 2) - i) / 3] * max_score for i in range(no_of_items)}

sim_mixed = rp.MFRM_Sim_Matrix(no_of_items=no_of_items, no_of_persons=2000, no_of_facet_elements=8, max_score=max_score, person_sd=4,
                               manual_raters={'Rater_1': flat, 'Rater_2': flat, 'Rater_3': flat, 'Rater_4': flat,
                                              'Rater_5': halo, 'Rater_6': halo, 'Rater_7': halo, 'Rater_8': halo},
                               seed=1)

generating_models = {'Rater_1': 'global', 'Rater_2': 'global', 'Rater_3': 'global', 'Rater_4': 'global',
                     'Rater_5': 'items', 'Rater_6': 'items', 'Rater_7': 'items', 'Rater_8': 'items'}

mfrm_mixed = rp.MFRM(sim_mixed)
mfrm_mixed.calibrate_matrix()   # calibrates the full matrix model once; everything else is derived from it, no refitting
mfrm_mixed.per_rater_model_selection(anchors=['Rater_1', 'Rater_2', 'Rater_3', 'Rater_4'], min_effect=0.43)

`rater_models` gives the assigned parameterisation per rater; `per_rater_model_selection_counts` summarises how many raters landed on each:

In [ ]:
mfrm_mixed.anchor_rater_models

In [ ]:
mfrm_mixed.anchor_per_rater_model_selection_counts

Compare against the true generating parameterisation for each rater:

In [ ]:
import pandas as pd

comparison = pd.DataFrame({'Generating': pd.Series(generating_models),
                           'Selected': mfrm_mixed.anchor_rater_models})
comparison['Correct'] = (comparison['Generating'] == comparison['Selected'])
comparison

With every rater assigned a representation, fit statistics and rater statistics for the mixed model can be generated exactly as for any single parameterisation by passing `model='mixed'`:

In [ ]:
mfrm_mixed.rater_stats_df(model='mixed', anchors=['Rater_1', 'Rater_2', 'Rater_3', 'Rater_4'], full=True, no_of_samples=200)
mfrm_mixed.rater_stats_mixed.head()

`per_rater_model_selection()` also takes `models=`, restricting the choice of models as above. This call keeps `test='AIC'` (the default) — under AIC or BIC, `models=` is a plain filter: the minimum is taken over whichever models are passed; for AIC specifically, `aic_sig_test` additionally compares the winner against whichever active model has the fewest parameters. When `test='LR'`, a backbone of `['global', 'items', 'thresholds', 'bivector']` must be included in the model set, since LR is strictly a test for nested model; the LR ladder's fork logic hardcodes those four models as its literal comparison targets (bivector vs items, bivector vs thresholds, items/thresholds vs global), so dropping one leaves a step with nothing to compare against — `'matrix'` and each of `'centrality'`/`'pseudo_halo'`/`'bistretch'` stay independently optional even under LR.

Restricting to `['global', 'items', 'thresholds', 'bivector']` here recovers the original five-representation family (as if `'matrix'` and the three stretch models did not exist), which is useful when they are not of interest for a given design, or to match a legacy analysis. Since this data was only ever generated under `'global'`/`'items'`, the recovered assignment should be unchanged:

In [ ]:
mfrm_mixed.per_rater_model_selection(anchors=['Rater_1', 'Rater_2', 'Rater_3', 'Rater_4'], min_effect=0.43,
                                      models=['global', 'items', 'thresholds', 'bivector'])
mfrm_mixed.anchor_rater_models